In [ ]:
#Import libraries

from datasets import load_dataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import json
import joblib
import mlflow
import mlflow.sklearn


from sklearn.model_selection import (train_test_split, RandomizedSearchCV,
                                     StratifiedKFold)
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             ConfusionMatrixDisplay)
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import (VarianceThreshold, SelectKBest,
                                       f_classif)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

In [ ]:
RANDOM_SEED = 42

In [ ]:
#load dataset from notebook 01
ds = load_dataset("Fllamber/TCGA-PANCAN-HiSeq-2770x20530")
ds.set_format(type='pandas')

df = ds['train'][:]

X = df.drop(columns=['cancer']) 
y = df['cancer']

#test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_SEED)

#fitting and validation splits
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=RANDOM_SEED)

Experiment wheter a reduction method improves the models compared to using the original features by selecting reduction='none', 'pca' or 'selection'.

In [ ]:
#Define function for model tuning
def tune_model(name, estimator, param_distributions,
               X_fit, y_fit, X_val, y_val,
               reduction="selection",
               n_iter=3):

    steps = []

    #Choose preprocessing method
    if reduction=="pca":

        steps.append(("scaler", StandardScaler()))
        steps.append(("pca", PCA(n_components=0.95)))

    elif reduction=="selection":

        steps.append(("remove_constant", VarianceThreshold(threshold=0.0)))
        steps.append(("scaler", StandardScaler()))
        steps.append(("select", SelectKBest(score_func=f_classif, k=1000)))

    elif reduction=="none":
        steps.append(("scaler", StandardScaler()))

    else:
        raise ValueError("reduction must be 'none', 'pca', or 'selection'")

    #ADd classifier after preprocessing
    steps.append(("model", estimator))
    
    pipeline = Pipeline(steps)

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOM_SEED
    )
    #Search for best hyperparameter combination
    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_distributions,
        n_iter=n_iter,
        scoring="f1_macro",
        cv=cv,
        random_state=RANDOM_SEED,
        n_jobs=1,
        verbose=1,
        refit=True,
        error_score="raise"
    )

    search.fit(X_fit, y_fit)

    best_model = search.best_estimator_

    y_val_pred_model = best_model.predict(X_val)

    accuracy_model = accuracy_score(y_val, y_val_pred_model)
    f1_macro_model = f1_score(y_val, y_val_pred_model, average='macro')

    print(f"{name}")
    print("Feature reduction:", reduction)
    print("Best parameters:", search.best_params_)
    print("Best CV Macro F1:", search.best_score_)
    print("Validation Accuracy:", accuracy_model)
    print("Validation Macro F1:", f1_macro_model)

    print("Validation classification report:")
    print(classification_report(y_val, y_val_pred_model))

    result_model = {
        "Model": name,
        "Feature reduction": reduction,
        "Best CV Macro F1": search.best_score_,
        "Validation Accuracy": accuracy_model,
        "Validation Macro F1": f1_macro_model,
        "Best Parameters": search.best_params_
    }

    return search, result_model

In [ ]:
#Define models and hyperparameters

models_to_tune = {
    "Random Forest": {
        "estimator": RandomForestClassifier(
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=1
        ),
        "params": {
            "model__n_estimators": [100, 200, 300],
            "model__max_depth": [10, 20, 30, None],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4, 8],
            "model__max_features": ["sqrt", 0.3, 0.5]
        }
    },

    "Extra Trees": {
        "estimator": ExtraTreesClassifier(
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=1
        ),
        "params": {
            "model__n_estimators": [100, 200, 300],
            "model__max_depth": [10, 20, 30, None],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4, 8],
            "model__max_features": ["sqrt", 0.3, 0.5]
        }
    },

    "LightGBM": {
        "estimator": LGBMClassifier(
            class_weight="balanced",
            random_state=RANDOM_SEED,
            verbosity=-1,
            n_jobs=1
        ),
        "params": {
            "model__n_estimators": [50, 100, 200, 300],
            "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
            "model__num_leaves": [15, 31, 50],
            "model__max_depth": [-1, 10, 20],
            "model__min_child_samples": [10, 20, 30, 50]
        }
    },

    "CatBoost": {
        "estimator": CatBoostClassifier(
            auto_class_weights="Balanced",
            random_seed=RANDOM_SEED,
            verbose=0,
            allow_writing_files=False,
            thread_count=1
        ),
        "params": {
            "model__iterations": [100, 200, 300, 500],
            "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
            "model__depth": [4, 5, 6, 8],
            "model__l2_leaf_reg": [1, 3, 5, 7]
        }
    },

    "XGBoost": {
        "estimator": XGBClassifier(
            objective="multi:softprob",
            eval_metric="mlogloss",
            tree_method="hist",
            random_state=RANDOM_SEED,
            n_jobs=1
        ),
        "params": {
            "model__n_estimators": [50, 100, 200, 300],
            "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
            "model__max_depth": [3, 4, 5, 7],
            "model__min_child_weight": [1, 3, 5],
            "model__subsample": [0.7, 0.8, 1.0],
            "model__colsample_bytree": [0.5, 0.7, 0.9]
        }
    }
}

In [ ]:
#Tune every model in the dictionary
all_searches = {}
all_results = []

for model_name, config in models_to_tune.items():

    search, result = tune_model(
        name=model_name,
        estimator=config["estimator"],
        param_distributions=config["params"],
        X_fit=X_fit,
        y_fit=y_fit,
        X_val=X_val,
        y_val=y_val,
        reduction="selection",
        n_iter=3
    )

    all_searches[model_name] = search
    all_results.append(result)

all_results_df = pd.DataFrame(all_results)

best_results_df = all_results_df.sort_values(
    by="Validation Macro F1",
    ascending=False).reset_index(drop=True)

print(best_results_df[
    [
        "Model",
        "Feature reduction",
        "Best CV Macro F1",
        "Validation Accuracy",
        "Validation Macro F1",
    ]
].round(4))

The results show that both Random Forest and Extra Tress perform very well on the dataset. Random Forest shows the best CV Macro F1 score, while Extra trees show slightly better validation accurancy and validation Macro-F1. Random Forest scored slighlty higher during cross-validation, but Extra Trees performed better on the separate validation set. 

The performance difference between Random Forest and Extra Trees was relatively very small. Random Forest is selected for final evaluation on the test set to show the performace on unseen data.

In [ ]:
#Compare validation predictions from the two models
rf_pred = all_searches["Random Forest"].best_estimator_.predict(X_val)
et_pred = all_searches["Extra Trees"].best_estimator_.predict(X_val)

#check where predictions differ
different = rf_pred != et_pred

comparison = pd.DataFrame({
    "Actual cancer": y_val.iloc[different].values,
    "Random Forest prediction": rf_pred[different],
    "Extra Trees prediction": et_pred[different]
})

print(comparison)

In [ ]:
#Evaluate selected model on the test set
final_model = all_searches["Random Forest"].best_estimator_

y_test_pred = final_model.predict(X_test)

print("Test accuracy:", round(accuracy_score(y_test, y_test_pred), 4))
print(
    "Test macro F1:", round(
    f1_score(y_test, y_test_pred, average="macro"), 4))
print(classification_report(y_test, y_test_pred))

The Random Forest model achieved an excellent result on the unseen test set, with 99.76% accuracy and a Macro F1-score of 0.9981 on the 831 test set samples. The classification report shows consistently high performance acros all five cancer types and suggests that the model classified almost all 831 samples from the test set corect.  

In [ ]:
#Visualize final test predictions
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_test_pred,
    cmap="Blues",
    xticks_rotation=45
)

plt.title("Random Forest — Test Set Confusion Matrix")
plt.tight_layout()
plt.show()

In the confusion matrix the rows represent the actual cancer types and the columns represent the predicted cancer types. Correct predictions are on the main diagonal. The off-diagonal values show which cancer typer where confused with other cancer type. 
The confusion matrix shows that two KIRC samples were incorrectly classified as BRCA, the remaining samples were classified correct. The results indicate that The tuned Random Forest model performed overall very well.


In [ ]:
#Save the fitted pipeline
joblib.dump(final_model, "random_forest_cancer_model.joblib")
print("Model savec")

In [ ]:
#save original feauter names and order
feature_names = X_fit.columns.tolist()

with open('feature_names.json', "w") as f:
    json.dump(feature_names, f)

print('Feature names saved')
len(feature_names)

In [ ]:
# Store MLflow experiments in root
project_root = os.path.abspath("..")

# Create SQLite database in project root
mlflow_db_path = os.path.join(project_root, "mlflow.db")
tracking_uri = f"sqlite:///{mlflow_db_path}"

mlflow.set_tracking_uri(tracking_uri)

# Create experiment
mlflow.set_experiment("Cancer-Classification")

print("Project root:", project_root)
print("MLflow database:", mlflow_db_path)
print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("MLflow experiment is ready")

In [ ]:
#Log each tuned models parameters and validation metric
for model_name, search in all_searches.items():

    # Get best fitted model
    best_model = search.best_estimator_

    # Calculate validation predictions
    val_predictions = best_model.predict(X_val)

    #Calculate validation metrics
    val_accuracy = accuracy_score(y_val, val_predictions)
    val_macro_f1 = f1_score(
        y_val,
        val_predictions,
        average="macro"
    )

    # Create MLflow run for model
    with mlflow.start_run(run_name=model_name):

        # Log the best hyperparameters
        mlflow.log_params(search.best_params_)

        # Log cross-validation and validation metrics
        mlflow.log_metric(
            "cv_macro_f1",
            search.best_score_
        )

        mlflow.log_metric(
            "validation_accuracy",
            val_accuracy
        )

        mlflow.log_metric(
            "validation_macro_f1",
            val_macro_f1
        )

        # Add info to identify experiment
        mlflow.set_tag("model_name", model_name)
        mlflow.set_tag("dataset", "TCGA-PANCAN-HiSeq")
        mlflow.set_tag(
            "feature_reduction",
            "selection"
        )

        # Save the fitted pipeline as an MLflow artifact
        mlflow.sklearn.log_model(
            sk_model=best_model,
            artifact_path="model",
            skops_trusted_types=[
        "sklearn.feature_selection._univariate_selection.f_classif"]
        )

        print(
            f"Logged {model_name}: "
            f"validation macro-F1 = {val_macro_f1:.4f}"
        )